In [1]:
import torch
import torchphysics as tp
import pandas as pd
import numpy as np
import torch.nn as nn
import matplotlib.pyplot as plt
import torch.nn.functional as F
from torch.utils.data import DataLoader
import pytorch_lightning as pl
import os
from pytorch_lightning import loggers as pl_loggers
# GAN macros
GPU="cpu"
GAN_weight=1 # Generator weight
L_x=3.6 # length of domain
N_x=200 # grid in x
N_x_sub=64
N_y=100 # grid in y
N_dists=1 # N 1d roughness for fake images during trainingN_epochs
N_epochs=2000 # N training iteration each epoch
GP_weight=10
Grid_data=True



/net/istmhome/users/hi224/Dokumente/Python/TorchPhysics/TP/lib/python3.8/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
X = tp.spaces.R1('x')
Y = tp.spaces.R1('y')
C = tp.spaces.R1("c") # number of case
#output
U = tp.spaces.R1('u')
V = tp.spaces.R1('v')
URMS = tp.spaces.R1('urms')
VRMS = tp.spaces.R1('vrms')
UV=tp.spaces.R1('uv')
P=tp.spaces.R1('p')

In [3]:
def self_sin(input):
    return input.sin()
def self_cos(input):
    return input.cos()

In [4]:
class ResidualBlock_1d(nn.Module):
    def __init__(self, in_channels, out_channels, stride = 1, downsample = None):
        super(ResidualBlock_1d, self).__init__()
        self.conv1 = nn.Sequential(
                        nn.Conv1d(in_channels, out_channels, kernel_size = 3, stride = stride, padding = 1),
                        nn.BatchNorm1d(out_channels),
                        nn.ReLU())
        self.conv2 = nn.Sequential(
                        nn.Conv1d(out_channels, out_channels, kernel_size = 3, stride = 1, padding = 1),
                        nn.BatchNorm1d(out_channels))
        self.downsample = downsample
        self.relu = nn.ReLU()
        self.out_channels = out_channels
        
    def forward(self, x):
        residual = x
        out = self.conv1(x)
        out = self.conv2(out)
        if self.downsample:
            residual = self.downsample(x)
        out += residual
        out = self.relu(out)
        return out

In [5]:
class ResNet1d(nn.Module):
    def __init__(self, block, layers,input_space,output_space,N_features,sigma_1=1,sigma_2=15):
        super(ResNet1d, self).__init__()
        self.inplanes = 64
        self.conv1 = nn.Sequential(
                        nn.Conv1d(1, 64, kernel_size = 7, stride = 2, padding = 3),
                        nn.BatchNorm1d(64),
                        nn.ReLU())
        self.maxpool = nn.MaxPool1d(kernel_size = 3, stride = 2, padding = 1)
        self.layer0 = self._make_layer(block, 64, layers[0], stride = 1)
        self.layer1 = self._make_layer(block, 64, layers[1], stride = 2)
        self.layer2 = self._make_layer(block, 128, layers[2], stride = 2)
        self.layer3 = self._make_layer(block, 128, layers[3], stride = 2)
        self.avgpool = nn.AvgPool1d(2, stride=0)
        #self.act_binary=nn.Sigmoid()
        #self.fc_Res = nn.Linear(2048, 2)

        self.W_1 = torch.tensor(torch.randn(input_space.dim , N_features //2, dtype=torch.float32,device=GPU) * sigma_1, dtype=torch.float32, requires_grad=False)
        self.W_2 = torch.tensor(torch.randn(input_space.dim , N_features //2, dtype=torch.float32,device=GPU) * sigma_2, dtype=torch.float32, requires_grad=False)
        self.register_buffer("selfW1", self.W_1, persistent=False)
        self.register_buffer("selfW2", self.W_2, persistent=False)
        self.output_space=output_space
        self.fc1_l=nn.Linear(in_features=N_features,out_features=150)
        self.fc2_l=nn.Linear(in_features=150,out_features=150)
        self.fc3_l=nn.Linear(in_features=150,out_features=150)
        ###
        self.fc1_r=nn.Linear(in_features=N_features,out_features=150)
        self.fc2_r=nn.Linear(in_features=150,out_features=150)
        self.fc3_r=nn.Linear(in_features=150,out_features=150)


        self.fc_combo1=nn.Linear(in_features=2944+300,out_features=2048)
        self.fc_combo2=nn.Linear(in_features=2048,out_features=2048)
        self.fc_combo3=nn.Linear(in_features=2048,out_features=2048)
        self.fc_combo4=nn.Linear(in_features=2048,out_features=2048)
        self.fc_combo5=nn.Linear(in_features=2048,out_features=2048)

        self.out=nn.Linear(in_features=2048,out_features=output_space.dim)
    def _make_layer(self, block, planes, blocks, stride=1):
        downsample = None
        if stride != 1 or self.inplanes != planes:
            
            downsample = nn.Sequential(
                nn.Conv1d(self.inplanes, planes, kernel_size=1, stride=stride),
                nn.BatchNorm1d(planes),
            )
        layers = []
        layers.append(block(self.inplanes, planes, stride, downsample))
        self.inplanes = planes
        for i in range(1, blocks):
            layers.append(block(self.inplanes, planes))

        return nn.Sequential(*layers)
    
    
    def forward(self,x,t):
        x = self.conv1(x)
        x = self.maxpool(x)
        x = self.layer0(x)
        x = self.layer1(x)
        x = self.layer2(x)
        x = self.layer3(x)

    #x = self.avgpool(x)
        x = x.view(x.size(0), -1)
        #x = nn.ReLU(self.fc_Res(x))
        t=t.as_tensor[:,0:2]
        t_1=torch.concat((self_sin(torch.matmul(t,self.W_1)),self_cos(torch.matmul(t,self.W_1))),1)
        t_1=self_sin(self.fc1_l(t_1))
        t_1=self_sin(self.fc2_l(t_1))
        t_1=self_sin(self.fc3_l(t_1))
        t_2=torch.concat((self_sin(torch.matmul(t,self.W_2)),self_cos(torch.matmul(t,self.W_2))),1)
        t_2=self_sin(self.fc1_r(t_2))
        t_2=self_sin(self.fc2_r(t_2))
        t_2=self_sin(self.fc3_r(t_2))

        t=torch.concat((t_1,t_2,x),1)
        t=self_sin(self.fc_combo1(t))
        t=self_sin(self.fc_combo2(t))
        t=self_sin(self.fc_combo3(t))
        t=self_sin(self.fc_combo4(t))
        t=self_sin(self.fc_combo5(t))

        t=self.out(t)

        return tp.problem.spaces.Points(t, self.output_space)
#model = ResNet1d(ResidualBlock_1d,[2,2,2,2],input_space=X*Y,output_space=U*V*URMS*VRMS*UV*P,N_features=300).to(GPU)
#disc=ResNet(ResidualBlock_1d, [2, 2, 2, 2]).to(GPU)

In [6]:
model=torch.load("PIAN_log_sum_logarithmic_full_reduced1dCNN.pt",map_location=torch.device('cpu'))

In [7]:
DF_DIST=pd.read_csv("../../Data/dist_2289_1Ds.csv",header=None)
x_corr=DF_DIST.iloc[0].to_numpy()
dist=DF_DIST.iloc[1:].to_numpy()
x_corr_new=np.linspace(min(x_corr),max(x_corr),N_x)
dist_new=np.zeros((len(dist),N_x))
for i in range(len(dist)):
    dist_new[i,:]=np.interp(x_corr_new,x_corr,dist[i])
x_corr_new=torch.tensor(x_corr_new,requires_grad=False, dtype=torch.float32,device=GPU)
x_corr=torch.tensor(x_corr,requires_grad=False, dtype=torch.float32,device=GPU)
dist_new=torch.tensor(dist_new,requires_grad=False, dtype=torch.float32,device=GPU)
dist=torch.tensor(dist, requires_grad=False,dtype=torch.float32,device=GPU)

In [8]:
def produce_sample(x_max,y_max,c,other_Constrains,resolution=100):
    list_x=np.linspace(0,x_max,200)
    list_y=np.linspace(0,y_max,resolution)
    XM,YM=np.meshgrid(list_x,list_y)
    model_value_x=np.zeros(XM.shape)
    model_value_y=np.zeros(XM.shape)    
    model_value_urms=np.zeros(XM.shape)
    model_value_uv=np.zeros(XM.shape)
    model_value_p=np.zeros(XM.shape)
    dist_1d=dist[c][None,None,:]
    for i in range(len(list_x)):
        for j in range(len(list_y)):
            coords = torch.tensor([[list_x[i],list_y[j],c]], dtype=torch.float32,device=GPU)
            model_value_x[j,i] = model(dist_1d,tp.spaces.Points(coords, X*Y*C)).as_tensor[0,0]
            model_value_y[j,i] = model(dist_1d,tp.spaces.Points(coords, X*Y*C)).as_tensor[0,1]
            model_value_urms[j,i] = model(dist_1d,tp.spaces.Points(coords, X*Y*C)).as_tensor[0,2]
            model_value_uv[j,i] = model(dist_1d,tp.spaces.Points(coords, X*Y*C)).as_tensor[0,4]
            model_value_p[j,i] = model(dist_1d,tp.spaces.Points(coords, X*Y*C)).as_tensor[0,5]
    model_value_urms=np.transpose(model_value_urms)
    model_value_uv=np.transpose(model_value_uv)
    model_value_x=np.transpose(model_value_x)
    model_value_y=np.transpose(model_value_y)
    model_value_p=np.transpose(model_value_p)
    #plt.contourf(np.linspace(0,3.6,200),np.linspace(0,1,resolution),model_value_urms,levels=10,origin='lower')    
    #plt.colorbar()
    #plt.plot(x_corr.cpu(),dist[c].cpu())
    return model_value_x, model_value_y, model_value_urms, model_value_uv, model_value_p

In [ ]:
dist_c=0
model_values=produce_sample(3.6,1,dist_c,100)


In [ ]:
model_values[0].shape
plt.figure(figsize=(10,5))
#levels = np.linspace(-2.2, 2.2, 11)
plt.contourf(np.linspace(0,3.6,200),np.linspace(0,1,100),model_values[4].T,levels=5,extent=[0, 3.6, 0, 1],origin='lower')
plt.colorbar()
plt.plot(x_corr.cpu(),dist[dist_c].cpu())
plt.xlim(0,3.6)

In [ ]:
model_values[0].shape
plt.figure(figsize=(10,5))
levels = np.linspace(-2.2, 2.2, 21)
plt.contourf(np.linspace(0,3.6,200),np.linspace(0,1,100),model_values[2].T,levels=levels,extent=[0, 3.6, 0, 1],origin='lower')
plt.colorbar()
plt.plot(x_corr.cpu(),dist[dist_c].cpu())
plt.xlim(0,3.6)
plt.savefig("predictedurms_2D.png")

In [ ]:
DF_Data=pd.read_csv("../../Data/Flow_3d_mesh_2289_6_X200Y100_half.csv")
N_c=len(DF_Data["c"].unique()) # number of available training data
Data_Pinn=torch.permute(torch.tensor(DF_Data[["U","V","urms","vrms","uv"]].to_numpy(),dtype=torch.float32,device=GPU).reshape((N_c,N_x,N_y,5)),(0,3,1,2))


In [ ]:
plt.figure(figsize=(10,5))
levels = np.linspace(-2.2, 2.2, 21)
plt.contourf(np.linspace(0,3.6,200),np.linspace(0,1,100),Data_Pinn[dist_c,2,:,:].T,levels=levels,extent=[0, 3.6, 0, 1],origin='lower')
plt.colorbar()
plt.plot(x_corr.cpu(),dist[dist_c].cpu())
plt.xlim(0,3.6)
plt.savefig("truthurms_2D.png")

In [ ]:
component=2
residual=abs(torch.tensor(model_values)[2].T-Data_Pinn[dist_c,2,:,:].T)/abs(Data_Pinn[dist_c,2,:,:].T)
levels_err = np.linspace(0, 0.5, 5)
plt.figure(figsize=(20,5))
plt.contourf(np.linspace(0,3.6,200),np.linspace(0,1,100),residual,levels=levels_err,origin="lower")
plt.colorbar()
plt.plot(x_corr.cpu(),dist[dist_c].cpu())
plt.plot([0,3.6],[0.10,0.10])

In [ ]:
residual.shape

In [ ]:
meanerr=residual[10:,:].mean()
meanerr

In [ ]:
Uprofile=np.mean(model_values[2],axis=0)
Uprofile_real=torch.mean(Data_Pinn[0,2,:,:],dim=0)
plt.plot(Uprofile,label="predicted")
plt.plot(Uprofile_real,label="truth")
plt.legend()
plt.title("urms")
plt.xscale("log")
plt.savefig("compare_urms.png")

In [ ]:
Uprofile=np.mean(model_values[0],axis=0)
Uprofile_real=torch.mean(Data_Pinn[0,0,:,:],dim=0)
plt.plot(Uprofile,label="predicted")
plt.plot(Uprofile_real,label="truth")
plt.legend()
plt.xscale("log")
plt.title("Umean")
plt.savefig("compare_U.png")

In [ ]:
Uprofile=np.mean(model_values[3],axis=0)
Uprofile_real=torch.mean(Data_Pinn[0,4,:,:],dim=0)
plt.plot(Uprofile,label="predicted")
plt.plot(Uprofile_real,label="truth")
plt.legend()
plt.title("uv")
plt.xscale("log")
plt.savefig("compare_uv.png")